In [1]:
# Task 1 Character Frequency Count and TreeNode Creation
# Task 1.1: Define the TreeNode Data Structure 
class TreeNode:
    def __init__(self, freq, value=None, left=None, right=None):
        self.freq = freq
        self.value = value
        self.left = left
        self.right = right
        
    def __repr__(self):
        return f"{self.value}({self.freq})"

# Task 1.2: Implementation the function
message = 'Compression is cool'

def getFrequenciesTreeNodes(message):
    message = message.lower()
    
    freq_dict = {}
    for char in message:
        if char in freq_dict:
            freq_dict[char] += 1
        else:
            freq_dict[char] = 1
            
    node_list = []
    for char, count in freq_dict.items():
        new_node = TreeNode(freq=count, value=char)
        node_list.append(new_node)
        
    return node_list
    
print(getFrequenciesTreeNodes(message))

[c(2), o(4), m(1), p(1), r(1), e(1), s(3), i(2), n(1),  (2), l(1)]


In [2]:
# Task 2: Sorting

def mergeSort(treeNodes, left, right):
    # stop condition: if left >= right, array is already sorted/atomic
    if left < right:
        # find the midpoint to split the array
        mid = left + (right - left) // 2
        
        # recursively sort the left and right halves
        mergeSort(treeNodes, left, mid)
        mergeSort(treeNodes, mid + 1, right)
        
        # merge the sorted halves back together
        merge(treeNodes, left, mid, right)

def merge(treeNodes, left, mid, right):
    # get sizes of the two subarrays
    n1 = mid - left + 1
    n2 = right - mid
    
    # create temp arrays for left and right
    L = [None] * n1
    R = [None] * n2
    
    # copy data into the temp arrays
    for i in range(n1):
        L[i] = treeNodes[left + i]
    for j in range(n2):
        R[j] = treeNodes[mid + 1 + j]
        
    # merge temp arrays back into the main treeNodes array
    i = 0     # pointer for L
    j = 0     # pointer for R
    k = left  # pointer for the main array
    
    while i < n1 and j < n2:
        # Note: sorting in decreasing order based on frequency
        if L[i].freq >= R[j].freq:
            treeNodes[k] = L[i]
            i += 1
        else:
            treeNodes[k] = R[j]
            j += 1
        k += 1
        
    # copy any leftover elements from L (if any)
    while i < n1:
        treeNodes[k] = L[i]
        i += 1
        k += 1
        
    # copy any leftover elements from R (if any)
    while j < n2:
        treeNodes[k] = R[j]
        j += 1
        k += 1

In [3]:
# Task 3: Creating a singular linked list of TreeNodes
class Node:
    def __init__(self, treeNode, next=None):
        self.treeNode = treeNode
        self.next = next

class LinkedList:
    def __init__(self, head=None):
        self.head = head
        
    def add(self, node):
        node.next = self.head
        self.head = node
        
    def insert(self, node):
        # If the list is empty, just set it as head
        if self.head is None:
            self.head = node
            return
            
        # Helper to check if node_a should go before node_b
        def should_insert_before(node_a, node_b):
            # Sort by frequency first
            if node_a.treeNode.freq != node_b.treeNode.freq:
                return node_a.treeNode.freq < node_b.treeNode.freq
            # If frequencies tie, sort alphabetically by value
            val_a = node_a.treeNode.value if node_a.treeNode.value else ""
            val_b = node_b.treeNode.value if node_b.treeNode.value else ""
            return val_a < val_b

        # Check if we need to insert before the current head
        if should_insert_before(node, self.head):
            node.next = self.head
            self.head = node
            return
            
        # Traverse to find the correct insertion point
        current = self.head
        while current.next is not None and not should_insert_before(node, current.next):
            current = current.next
            
        # Insert the node
        node.next = current.next
        current.next = node
        
    def removeTwoNodesAtFront(self):
        # Need at least two nodes to remove
        if self.head is None or self.head.next is None:
            return None, None
            
        node1 = self.head
        node2 = self.head.next
        
        # Update head to the 3rd node
        self.head = node2.next
        
        # Detach the removed nodes completely to prevent bugs
        node1.next = None
        node2.next = None
        
        return node1, node2
        
    def __len__(self):
        count = 0
        current = self.head
        while current is not None:
            count += 1
            current = current.next
        return count
        
    def __str__(self):
        result = []
        current = self.head
        while current is not None:
            val = current.treeNode.value
            # Format space as underscore for printing
            if val == ' ':
                val = '_'
            result.append(f"{val}:{current.treeNode.freq}")
            current = current.next
        return "->".join(result)

In [4]:
# 1. Get the initial list of TreeNodes
nodes = getFrequenciesTreeNodes("Compression is cool")

# 2. Sort the list in decreasing order
mergeSort(nodes, 0, len(nodes) - 1)

# 3. Add to linked list sequentially
llist = LinkedList()
for tree_node in nodes:
    # Wrap the TreeNode in a linked list Node before adding
    llist.add(Node(tree_node))

# 4. Check the output
print(llist)

l:1->n:1->e:1->r:1->p:1->m:1->_:2->i:2->c:2->s:3->o:4


In [5]:
# Task 4: Building the Huffman Tree

def get_pretty_tree(node, level=0):
    # Generates a string representation of the tree using pre-order traversal
    # This closely matches the top-down indentation style shown in the assignment.
    if node is None:
        return ""
        
    # Format the current node
    val = node.value if node.value != ' ' else '_'
    
    # Use indentation to show the level/depth of the tree
    result = "  " * level + f"{val}({node.freq})\n"
    
    # Recursively format left and right children
    result += get_pretty_tree(node.left, level + 1)
    result += get_pretty_tree(node.right, level + 1)
    
    return result

def display(root):
    # Renders the entire tree structure
    if root is None:
        print("Empty Tree")
        return
    print(get_pretty_tree(root))

def buildHuffmanTree(myList):
    # Constructs a Huffman tree based on character frequencies
    # Loop until only one node the root remains in the linked list
    while len(myList) > 1:
        # 1. Select and Merge: Get the two nodes with the lowest frequencies
        node1, node2 = myList.removeTwoNodesAtFront()
        
        t1 = node1.treeNode
        t2 = node2.treeNode
        
        # Calculate new frequency and concatenate values
        new_freq = t1.freq + t2.freq
        new_val = t1.value + t2.value
        
        # Create a new parent TreeNode. 
        # t1 goes left, t2 goes right.
        parent_tree_node = TreeNode(freq=new_freq, value=new_val, left=t1, right=t2)
        
        # 2. Iterate: Insert the new parent back into the sorted linked list
        new_ll_node = Node(parent_tree_node)
        myList.insert(new_ll_node)
        
    # 3. Establish the Root: The final remaining node is the root
    if myList.head is not None:
        return myList.head.treeNode
    return None

In [6]:
# Task 5: Encode Message to Binary String
def treeTraversal(root, c):
    # Base case: empty node
    if root is None:
        return None
        
    # If we hit a leaf node, check if it holds our target character
    if root.left is None and root.right is None:
        if root.value == c:
            return ""  # Found it, start building the path
        else:
            return None
            
    # Go left append '0' if the character is found down this path
    left_path = treeTraversal(root.left, c)
    if left_path is not None:
        return "0" + left_path
        
    # Go right append '1' if the character is found down this path
    right_path = treeTraversal(root.right, c)
    if right_path is not None:
        return "1" + right_path
        
    # Not found in this branch
    return None

def encodeToBinaryString(root, message):
    encoded_str = ""
    # Make sure to convert the input message to lowercase
    message = message.lower()
    
    for char in message:
        # Find the binary path for each character and add it to our result
        path = treeTraversal(root, char)
        if path is not None:
            encoded_str += path
            
    return encoded_str

In [7]:
# Task 6: Decode Message
def decodeMessage(root, binary):
    decoded_str = ""
    current = root
    
    for bit in binary:
        # Move left for '0', right for '1'
        if bit == '0':
            current = current.left
        else:
            current = current.right
            
        # Check if we reached a leaf node
        if current.left is None and current.right is None:
            # Append the character to our result
            decoded_str += current.value
            # Reset current pointer back to root for the next character
            current = root
            
    return decoded_str

In [8]:
# 1. Setup & Sort
nodes = getFrequenciesTreeNodes("Compression is cool")
mergeSort(nodes, 0, len(nodes) - 1)

# 2. Build Linked List
llist = LinkedList()
for tree_node in nodes:
    llist.add(Node(tree_node))

# 3. Build Tree
root = buildHuffmanTree(llist)

# 4. Encode
original_message = "Compression is cool"
binary_string = encodeToBinaryString(root, original_message)
print("Encoded:", binary_string)

# 5. Decode
decoded_message = decodeMessage(root, binary_string)
print("Decoded:", decoded_message)

# 6. Verify
print("Success?", original_message.lower() == decoded_message)

Encoded: 001100111011011111111101101100001001011110000110111000110100100
Decoded: compression is cool
Success? True
